# Adjusted Transparent Lifecycle Order

This notebook creates an adjusted transparent lifecycle-order table without changing `rq1_transparent_lifecycle_order_with_report_dates.ipynb`.

Adjustments implemented here:
- Keep the original full `D H:M:S` timestamps for inspection.
- For ordering, treat events on the same calendar date as tied and use this tie order: `Report`, `Fix`, `Release`, `Disclosure`.
- If `Report` is last and the report type is `github advisory`, relabel that lifecycle event as `GAD` instead of `Report`.
- Recompute the lifecycle summary after these adjustments.

In [142]:
from pathlib import Path

import pandas as pd

In [143]:
DATA_DIR = Path('.')

PRESENT_CSV = DATA_DIR / 'links_in_message_present_created_dates.csv'
RETRY_OK_CSV = DATA_DIR / 'links_in_message_present_unavailable_retry_recovered_ok.csv'
RECOVERED_CSV = DATA_DIR / 'links_in_message_recovered_created_dates.csv'

FIX_RELEASE_PATH = Path('../external_release_nvd/fix_releases_from_patch_data_local_server.csv')
NVD_PATH = Path('../external_release_nvd/cve_NVD_disclosure_dates.csv')

present_df = pd.read_csv(PRESENT_CSV)
retry_ok_df = pd.read_csv(RETRY_OK_CSV)
recovered_df = pd.read_csv(RECOVERED_CSV)
fix_release = pd.read_csv(FIX_RELEASE_PATH)
nvd = pd.read_csv(NVD_PATH)

print('present report-link rows:', present_df.shape)
print('retry recovered rows:', retry_ok_df.shape)
print('recovered report-link rows:', recovered_df.shape)
print('fix/release rows:', fix_release.shape)
print('nvd rows:', nvd.shape)

present report-link rows: (329, 16)
retry recovered rows: (1, 25)
recovered report-link rows: (142, 24)
fix/release rows: (832, 6)
nvd rows: (832, 2)


## Normalize Transparent Report Dates

In [144]:
REPORTING_TYPE_ORDER = [
    'github pull request',
    'github issue',
    'apache issue tracker',
    'github commit',
    'github advisory',
    'other',
]


def clean_string(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    return value if value else pd.NA


def reporting_type_for_table(value, row=None):
    value = str(value).strip().lower()
    link_text = ''
    if row is not None:
        link_candidates = [
            row.get('first_link_in_message', pd.NA),
            row.get('first_link', pd.NA),
            row.get('recovered_first_link', pd.NA),
            row.get('recovered_link', pd.NA),
            row.get('retry_link', pd.NA),
        ]
        link_text = ' '.join(str(x).lower() for x in link_candidates if not pd.isna(x))

    if 'github pull' in value:
        return 'github pull request'
    if 'github issue' in value:
        return 'github issue'
    if 'github commit' in value:
        return 'github commit'
    if 'github advisory' in value:
        return 'github advisory'
    if 'apache issue tracker' in value:
        return 'apache issue tracker'
    if 'jira' in value or 'bugzilla' in value or 'issue tracker' in value:
        if 'issues.apache.org' in link_text or 'bz.apache.org' in link_text:
            return 'apache issue tracker'
        return 'other'
    return 'other'


def derive_reporting_type(row):
    existing = clean_string(row.get('type_of_reporting', pd.NA))
    if not pd.isna(existing):
        return reporting_type_for_table(existing, row)

    candidates = [
        row.get('recovered_first_link_type', pd.NA),
        row.get('mined_recovered_link_type', pd.NA),
        row.get('retry_mined_link_type', pd.NA),
        row.get('mined_link_type', pd.NA),
        row.get('first_link_type_for_created_date', pd.NA),
    ]
    text = ' '.join(str(x).lower() for x in candidates if not pd.isna(x))
    return reporting_type_for_table(text, row)


def normalize_present(df):
    out = pd.DataFrame({
        'CVE_ID': df['CVE_ID'],
        'repository': df['repository'],
        'PATCH': df['PATCH'],
        'reporting_link': df['first_link_in_message'].fillna(df.get('first_link')),
        'Report Date': df['link_created_at'],
        'report_created_status': df['link_created_date_status'],
        'report_created_source': df['link_created_date_source'],
        'created_date_kind': df['created_date_kind'],
        'report_source': 'links in message present',
    })
    out['type_of_reporting'] = df.apply(derive_reporting_type, axis=1)
    return out


def normalize_retry(df):
    out = pd.DataFrame({
        'CVE_ID': df['CVE_ID'],
        'repository': df['repository'],
        'PATCH': df['PATCH'],
        'reporting_link': df['first_link_in_message'],
        'Report Date': df['retry_link_created_at'],
        'report_created_status': df['retry_link_created_date_status'],
        'report_created_source': df['retry_link_created_date_source'],
        'created_date_kind': df['retry_created_date_kind'],
        'report_source': 'retry recovered present link',
    })
    out['type_of_reporting'] = df.apply(derive_reporting_type, axis=1)
    return out


def normalize_recovered(df):
    out = pd.DataFrame({
        'CVE_ID': df['CVE_ID'],
        'repository': df['repository'],
        'PATCH': df['PATCH'],
        'reporting_link': df['recovered_link'].fillna(df['recovered_first_link']),
        'Report Date': df['recovered_link_created_at'],
        'report_created_status': df['recovered_link_created_date_status'],
        'report_created_source': df['recovered_link_created_date_source'],
        'created_date_kind': df['created_date_kind'],
        'report_source': 'links recovered from commit page',
    })
    out['type_of_reporting'] = df.apply(derive_reporting_type, axis=1)
    return out


report_df_all = pd.concat([
    normalize_present(present_df),
    normalize_retry(retry_ok_df),
    normalize_recovered(recovered_df),
], ignore_index=True)

# The retry-recovered row is also represented in the updated present file. Deduplicate by reporting link.
report_df_all = (
    report_df_all
    .drop_duplicates(subset=['CVE_ID', 'repository', 'PATCH', 'reporting_link'], keep='first')
    .reset_index(drop=True)
)

report_df = report_df_all[
    report_df_all['report_created_status'].eq('ok')
    & report_df_all['Report Date'].notna()
].copy()
report_df['Report Date'] = pd.to_datetime(report_df['Report Date'], errors='coerce', utc=True).dt.tz_convert(None)
report_df = report_df.dropna(subset=['Report Date']).copy()

print('all transparent reporting cases:', len(report_df_all))
print('cases with mined Report Date:', len(report_df))
report_df['type_of_reporting'].value_counts().reindex(REPORTING_TYPE_ORDER, fill_value=0)

all transparent reporting cases: 471
cases with mined Report Date: 298


type_of_reporting
github pull request     116
github issue             71
apache issue tracker     61
github commit            25
github advisory          24
other                     1
Name: count, dtype: int64

## Add Fix, Release, and Disclosure Dates

In [145]:
lifecycle_dates = fix_release[fix_release['Oldest Tag Date'].astype(str).str.lower().ne('not found')].copy()
lifecycle_dates['Fix Date'] = pd.to_datetime(lifecycle_dates['Commit Date'], errors='coerce')
lifecycle_dates['Release Date'] = pd.to_datetime(lifecycle_dates['Oldest Tag Date'], errors='coerce')

lifecycle_dates = lifecycle_dates.merge(nvd[['CVE_ID', 'Published Date']], on='CVE_ID', how='left')
lifecycle_dates['Disclosure Date'] = pd.to_datetime(lifecycle_dates['Published Date'], errors='coerce')

# The lifecycle source is CVE-level, so join by CVE_ID to match rq1_lifecycle_order_table.ipynb.
transparent_event_df = report_df.merge(
    lifecycle_dates[[
        'CVE_ID',
        'PATCH',
        'G:A:V',
        'Oldest Tag',
        'Fix Date',
        'Release Date',
        'Disclosure Date',
        'Published Date',
    ]],
    on='CVE_ID',
    how='left',
    suffixes=('', '_lifecycle'),
)

print('transparent rows with report dates before lifecycle filtering:', len(transparent_event_df))
print('Missing Report Date:', transparent_event_df['Report Date'].isna().sum())
print('Missing Fix Date:', transparent_event_df['Fix Date'].isna().sum())
print('Missing Release Date:', transparent_event_df['Release Date'].isna().sum())
print('Missing Disclosure Date:', transparent_event_df['Disclosure Date'].isna().sum())

transparent rows with report dates before lifecycle filtering: 298
Missing Report Date: 0
Missing Fix Date: 13
Missing Release Date: 13
Missing Disclosure Date: 13


In [146]:
transparent_event_df = transparent_event_df.dropna(subset=[
    'Report Date',
    'Fix Date',
    'Release Date',
    'Disclosure Date',
]).copy()

release_before_fix_df = transparent_event_df[
    transparent_event_df['Release Date'] < transparent_event_df['Fix Date']
].copy()
transparent_event_df = transparent_event_df[
    transparent_event_df['Release Date'] >= transparent_event_df['Fix Date']
].copy()

print('Rows excluded because Release Date < Fix Date:', len(release_before_fix_df))
print('Rows available for adjusted transparent lifecycle order:', len(transparent_event_df))
print('Unique CVEs:', transparent_event_df['CVE_ID'].nunique())

Rows excluded because Release Date < Fix Date: 6
Rows available for adjusted transparent lifecycle order: 279
Unique CVEs: 279


## Adjust Lifecycle Order

The adjustment is targeted to the lifecycle patterns you inspected:
- Keep `Report -> Fix -> Release -> Disclosure` and `Report -> Fix -> Disclosure -> Release` in full `D H:M:S` order.
- For selected patterns where `Fix` and `Report` occur on the same calendar date, reorder the same-day pair as `Report` before `Fix`.
- For `Fix -> Release -> Report -> Disclosure`, if `Report`, `Fix`, and `Release` are on the same calendar date, reorder as `Report -> Fix -> Release -> Disclosure`.
- If a trailing `Report` is a GitHub advisory, relabel that final event as `GAD`.

Full timestamps are preserved for inspection; adjusted order is applied only to the specific cases above.

In [147]:
def format_dhms(series):
    return pd.to_datetime(series, errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')


for col in ['Report Date', 'Fix Date', 'Release Date', 'Disclosure Date']:
    transparent_event_df[f'{col} DHMS'] = format_dhms(transparent_event_df[col])
    transparent_event_df[f'{col} Day'] = pd.to_datetime(transparent_event_df[col], errors='coerce').dt.normalize()

# Manual date-level correction after inspecting the selected lifecycle-order data point.
# CVE-2021-45046 has report date day Dec 13, 2021. Treat the fix and release day as
# Dec 13, 2021 for adjusted lifecycle ordering so it falls under Report -> Fix -> Release -> Disclosure.
manual_same_day_mask = transparent_event_df['CVE_ID'].eq('CVE-2021-45046')
manual_same_day = pd.Timestamp('2021-12-13')
transparent_event_df.loc[manual_same_day_mask, 'Fix Date Day'] = manual_same_day
transparent_event_df.loc[manual_same_day_mask, 'Release Date Day'] = manual_same_day
transparent_event_df.loc[manual_same_day_mask, 'manual_day_adjustment_note'] = (
    'Manual adjusted-order day correction: Fix/Release treated as 2021-12-13 to match inspected report date day.'
)
manual_fix_report_same_day_mask = transparent_event_df['CVE_ID'].eq('CVE-2021-20289')
manual_fix_report_same_day = pd.Timestamp('2021-04-13')
transparent_event_df.loc[manual_fix_report_same_day_mask, 'Fix Date Day'] = manual_fix_report_same_day
transparent_event_df.loc[manual_fix_report_same_day_mask, 'Report Date Day'] = manual_fix_report_same_day
transparent_event_df.loc[manual_fix_report_same_day_mask, 'manual_day_adjustment_note'] = (
    'Manual adjusted-order confirmation: Fix/Report treated as same day 2021-04-13; adjusted to Disclosure -> Report -> Fix -> Release.'
)

manual_spring_oauth_mask = transparent_event_df['CVE_ID'].eq('CVE-2019-3778')
manual_spring_oauth_day = pd.Timestamp('2019-02-18')
transparent_event_df.loc[manual_spring_oauth_mask, 'Fix Date Day'] = manual_spring_oauth_day
transparent_event_df.loc[manual_spring_oauth_mask, 'Report Date Day'] = manual_spring_oauth_day
transparent_event_df.loc[manual_spring_oauth_mask, 'manual_day_adjustment_note'] = (
    'Manual adjusted-order correction: Fix/Report treated as same day 2019-02-18 after manual verification; adjusted to Report -> Fix -> Release -> Disclosure.'
)

transparent_event_df['manual_day_adjustment_note'] = transparent_event_df.get(
    'manual_day_adjustment_note', pd.Series(index=transparent_event_df.index, dtype='object')
).fillna('')


def original_dhms_order(row):
    events = [
        ('Report', row['Report Date']),
        ('Fix', row['Fix Date']),
        ('Release', row['Release Date']),
        ('Disclosure', row['Disclosure Date']),
    ]
    tie_breaker = {'Report': 0, 'Fix': 1, 'Release': 2, 'Disclosure': 3}
    return [name for name, _ in sorted(events, key=lambda item: (item[1], tie_breaker[item[0]]))]


def same_calendar_day(row, left, right):
    return row[f'{left} Date Day'] == row[f'{right} Date Day']


def adjusted_targeted_order(row):
    ordered_names = original_dhms_order(row)
    original_pattern = tuple(ordered_names)
    is_github_advisory = row['type_of_reporting'] == 'github advisory'

    # Keep these patterns in original D H:M:S order.
    if original_pattern in {
        ('Report', 'Fix', 'Release', 'Disclosure'),
        ('Report', 'Fix', 'Disclosure', 'Release'),
        ('Disclosure', 'Report', 'Fix', 'Release'),
        ('Report', 'Disclosure', 'Fix', 'Release'),
    }:
        return ordered_names

    # Fix and Report are the same calendar date; treat Report as first in that same-day pair.
    if original_pattern == ('Fix', 'Report', 'Release', 'Disclosure'):
        if same_calendar_day(row, 'Fix', 'Report'):
            return ['Report', 'Fix', 'Release', 'Disclosure']
        return ordered_names

    if original_pattern == ('Fix', 'Report', 'Disclosure', 'Release'):
        if same_calendar_day(row, 'Fix', 'Report'):
            return ['Report', 'Fix', 'Disclosure', 'Release']
        return ordered_names

    if original_pattern == ('Disclosure', 'Fix', 'Report', 'Release'):
        if same_calendar_day(row, 'Fix', 'Report'):
            return ['Disclosure', 'Report', 'Fix', 'Release']
        return ordered_names

    # If all four events fall on the same calendar date, use the study tie-order.
    if original_pattern == ('Disclosure', 'Fix', 'Release', 'Report'):
        if (
            same_calendar_day(row, 'Disclosure', 'Fix')
            and same_calendar_day(row, 'Disclosure', 'Release')
            and same_calendar_day(row, 'Disclosure', 'Report')
        ):
            return ['Report', 'Fix', 'Release', 'Disclosure']
        if is_github_advisory:
            return ['Disclosure', 'Fix', 'Release', 'GAD']
        return ordered_names

    # Report, Fix, and Release are the same calendar date; treat Report first, then Fix, then Release.
    if original_pattern == ('Fix', 'Release', 'Report', 'Disclosure'):
        if (
            same_calendar_day(row, 'Fix', 'Report')
            and same_calendar_day(row, 'Fix', 'Release')
        ):
            return ['Report', 'Fix', 'Release', 'Disclosure']
        return ordered_names

    # Trailing GitHub advisory is not a report/issue creation event; label it GAD.
    if original_pattern == ('Fix', 'Release', 'Disclosure', 'Report'):
        if is_github_advisory:
            return ['Fix', 'Release', 'Disclosure', 'GAD']
        return ordered_names

    if original_pattern == ('Fix', 'Disclosure', 'Release', 'Report'):
        if is_github_advisory:
            return ['Fix', 'Disclosure', 'Release', 'GAD']
        return ordered_names

    return ordered_names

original_columns = transparent_event_df.apply(original_dhms_order, axis=1, result_type='expand')
original_columns.columns = ['Original First', 'Original Second', 'Original Third', 'Original Fourth']

adjusted_columns = transparent_event_df.apply(adjusted_targeted_order, axis=1, result_type='expand')
adjusted_columns.columns = ['First', 'Second', 'Third', 'Fourth']

transparent_lifecycle_order_adjusted_df = pd.concat([
    transparent_event_df[[
        'CVE_ID',
        'repository',
        'PATCH',
        'PATCH_lifecycle',
        'type_of_reporting',
        'reporting_link',
        'created_date_kind',
        'Report Date',
        'Fix Date',
        'Release Date',
        'Disclosure Date',
        'Report Date DHMS',
        'Fix Date DHMS',
        'Release Date DHMS',
        'Disclosure Date DHMS',
        'Report Date Day',
        'Fix Date Day',
        'Release Date Day',
        'Disclosure Date Day',
        'Oldest Tag',
        'manual_day_adjustment_note',
    ]].reset_index(drop=True),
    original_columns.reset_index(drop=True),
    adjusted_columns.reset_index(drop=True),
], axis=1)

transparent_lifecycle_order_adjusted_df['Original pattern'] = list(zip(
    transparent_lifecycle_order_adjusted_df['Original First'],
    transparent_lifecycle_order_adjusted_df['Original Second'],
    transparent_lifecycle_order_adjusted_df['Original Third'],
    transparent_lifecycle_order_adjusted_df['Original Fourth'],
))
transparent_lifecycle_order_adjusted_df['Adjusted pattern'] = list(zip(
    transparent_lifecycle_order_adjusted_df['First'],
    transparent_lifecycle_order_adjusted_df['Second'],
    transparent_lifecycle_order_adjusted_df['Third'],
    transparent_lifecycle_order_adjusted_df['Fourth'],
))
transparent_lifecycle_order_adjusted_df['Pattern changed'] = (
    transparent_lifecycle_order_adjusted_df['Original pattern']
    != transparent_lifecycle_order_adjusted_df['Adjusted pattern']
)

# Manual removal requested after inspection: remove the single
# Fix -> Release -> Disclosure -> Report case from the adjusted analysis.
manual_exclusion_mask = transparent_lifecycle_order_adjusted_df['CVE_ID'].eq('CVE-2013-5679')
manual_excluded_adjusted_df = transparent_lifecycle_order_adjusted_df[manual_exclusion_mask].copy()
transparent_lifecycle_order_adjusted_df = transparent_lifecycle_order_adjusted_df[~manual_exclusion_mask].copy()

print('Manually excluded adjusted rows:', len(manual_excluded_adjusted_df))
transparent_lifecycle_order_adjusted_df.head()

Manually excluded adjusted rows: 1


,CVE_ID,repository,PATCH,PATCH_lifecycle,type_of_reporting,reporting_link,created_date_kind,Report Date,Fix Date,Release Date,...,Original Second,Original Third,Original Fourth,First,Second,Third,Fourth,Original pattern,Adjusted pattern,Pattern changed
0,CVE-2013-4600,alkacon/opencms-core,https://github.com/alkacon/opencms-core/commit...,https://github.com/alkacon/opencms-core/commit...,github issue,https://github.com/alkacon/opencms-core/issues...,created_at,2013-06-13 21:26:01,2013-06-20 15:32:02,2013-07-09 11:57:20,...,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False
1,CVE-2018-3831,elastic/elasticsearch,https://github.com/elastic/elasticsearch/commi...,https://github.com/elastic/elasticsearch/commi...,github pull request,https://github.com/elastic/elasticsearch/pull/...,created_at,2018-08-29 16:33:41,2018-08-29 16:31:56,2018-08-29 16:31:56,...,Release,Report,Disclosure,Report,Fix,Release,Disclosure,"(Fix, Release, Report, Disclosure)","(Report, Fix, Release, Disclosure)",True
2,CVE-2019-16943,FasterXML/jackson-databind,https://github.com/FasterXML/jackson-databind/...,https://github.com/FasterXML/jackson-databind/...,github issue,https://github.com/FasterXML/jackson-databind/...,created_at,2019-09-27 15:44:21,2019-09-29 12:12:38,2019-11-09 15:11:27,...,Fix,Disclosure,Release,Report,Fix,Disclosure,Release,"(Report, Fix, Disclosure, Release)","(Report, Fix, Disclosure, Release)",False
3,CVE-2021-37714,jhy/jsoup,https://github.com/jhy/jsoup/commit/d2c455c94a...,https://github.com/jhy/jsoup/commit/d2c455c94a...,github issue,https://github.com/jhy/jsoup/issues/1613,created_at,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,...,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False
4,CVE-2020-36282,rabbitmq/rabbitmq-jms-client,https://github.com/rabbitmq/rabbitmq-jms-clien...,https://github.com/rabbitmq/rabbitmq-jms-clien...,github issue,https://github.com/rabbitmq/rabbitmq-jms-clien...,created_at,2020-11-02 10:38:12,2020-11-02 11:39:24,2020-11-03 09:27:56,...,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False


In [148]:
transparent_lifecycle_summary_adjusted = (
    transparent_lifecycle_order_adjusted_df
    .groupby(['First', 'Second', 'Third', 'Fourth'])
    .size()
    .reset_index(name='Count')
)
transparent_lifecycle_summary_adjusted['%'] = (
    transparent_lifecycle_summary_adjusted['Count']
    / len(transparent_lifecycle_order_adjusted_df)
    * 100
).round(2)
transparent_lifecycle_summary_adjusted = transparent_lifecycle_summary_adjusted.sort_values(
    'Count', ascending=False
).reset_index(drop=True)

transparent_lifecycle_summary_adjusted

,First,Second,Third,Fourth,Count,%
0,Report,Fix,Release,Disclosure,216,77.70
1,Report,Fix,Disclosure,Release,26,9.35
2,Disclosure,Report,Fix,Release,13,4.68
3,Fix,Release,Disclosure,GAD,13,4.68
4,Report,Disclosure,Fix,Release,6,2.16
5,Disclosure,Fix,Release,GAD,2,0.72
6,Fix,Disclosure,Release,GAD,2,0.72


## Adjustment Audit

This table shows which original full-timestamp (`D H:M:S`) patterns changed after date-level ordering and GitHub advisory relabeling.

In [149]:
adjustment_audit = (
    transparent_lifecycle_order_adjusted_df
    .groupby(['Original First', 'Original Second', 'Original Third', 'Original Fourth', 'First', 'Second', 'Third', 'Fourth'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
    .reset_index(drop=True)
)
adjustment_audit['Changed'] = (
    (adjustment_audit['Original First'] != adjustment_audit['First'])
    | (adjustment_audit['Original Second'] != adjustment_audit['Second'])
    | (adjustment_audit['Original Third'] != adjustment_audit['Third'])
    | (adjustment_audit['Original Fourth'] != adjustment_audit['Fourth'])
)
adjustment_audit

,Original First,Original Second,Original Third,Original Fourth,First,Second,Third,Fourth,Count,Changed
0,Report,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,189,False
1,Report,Fix,Disclosure,Release,Report,Fix,Disclosure,Release,23,False
2,Fix,Report,Release,Disclosure,Report,Fix,Release,Disclosure,16,True
3,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,GAD,13,True
4,Disclosure,Report,Fix,Release,Disclosure,Report,Fix,Release,12,False
5,Fix,Release,Report,Disclosure,Report,Fix,Release,Disclosure,10,True
6,Report,Disclosure,Fix,Release,Report,Disclosure,Fix,Release,6,False
7,Fix,Report,Disclosure,Release,Report,Fix,Disclosure,Release,3,True
8,Disclosure,Fix,Release,Report,Disclosure,Fix,Release,GAD,2,True
9,Fix,Disclosure,Release,Report,Fix,Disclosure,Release,GAD,2,True


## Inspect Data Points by Adjusted Lifecycle Order

Change `ORDER_TO_INSPECT` to inspect full rows for any adjusted lifecycle order.

In [163]:
INSPECTION_COLUMNS = [
    'CVE_ID',
    'repository',
    'type_of_reporting',
    'Original First',
    'Original Second',
    'Original Third',
    'Original Fourth',
    'First',
    'Second',
    'Third',
    'Fourth',
    'Report Date DHMS',
    'Fix Date DHMS',
    'Release Date DHMS',
    'Disclosure Date DHMS',
    'Report Date Day',
    'Fix Date Day',
    'Release Date Day',
    'Disclosure Date Day',
    'reporting_link',
    'created_date_kind',
    'Oldest Tag',
    'manual_day_adjustment_note',
    'PATCH',
    'PATCH_lifecycle',
]

adjusted_lifecycle_order_data_points = {
    order_key: group[INSPECTION_COLUMNS].sort_values(
        ['Report Date Day', 'Fix Date Day', 'Release Date Day', 'Disclosure Date Day', 'CVE_ID']
    ).reset_index(drop=True)
    for order_key, group in transparent_lifecycle_order_adjusted_df.groupby(
        ['First', 'Second', 'Third', 'Fourth'], dropna=False
    )
}

print('Available adjusted lifecycle orders:')
for _, row in transparent_lifecycle_summary_adjusted.iterrows():
    key = (row['First'], row['Second'], row['Third'], row['Fourth'])
    print(f'{key}: {row["Count"]} rows')

ORDER_TO_INSPECT = ('Fix', 'Release', 'Disclosure', 'GAD')

selected_adjusted_lifecycle_order_df = adjusted_lifecycle_order_data_points.get(
    ORDER_TO_INSPECT,
    pd.DataFrame(columns=INSPECTION_COLUMNS),
)
print()
print('Selected adjusted lifecycle order:', ORDER_TO_INSPECT)
print('Rows:', len(selected_adjusted_lifecycle_order_df))
selected_adjusted_lifecycle_order_df

Available adjusted lifecycle orders:
('Report', 'Fix', 'Release', 'Disclosure'): 216 rows
('Report', 'Fix', 'Disclosure', 'Release'): 26 rows
('Disclosure', 'Report', 'Fix', 'Release'): 13 rows
('Fix', 'Release', 'Disclosure', 'GAD'): 13 rows
('Report', 'Disclosure', 'Fix', 'Release'): 6 rows
('Disclosure', 'Fix', 'Release', 'GAD'): 2 rows
('Fix', 'Disclosure', 'Release', 'GAD'): 2 rows

Selected adjusted lifecycle order: ('Fix', 'Release', 'Disclosure', 'GAD')
Rows: 13


,CVE_ID,repository,type_of_reporting,Original First,Original Second,Original Third,Original Fourth,First,Second,Third,...,Report Date Day,Fix Date Day,Release Date Day,Disclosure Date Day,reporting_link,created_date_kind,Oldest Tag,manual_day_adjustment_note,PATCH,PATCH_lifecycle
0,CVE-2019-0232,apache/tomcat,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2019-04-18,2019-04-10,2019-04-10,2019-04-15,https://github.com/advisories/GHSA-8vmx-qmch-mpqg,published_at,9.0.18,,https://github.com/apache/tomcat/commit/4b244d...,https://github.com/apache/tomcat/commit/4b244d...
1,CVE-2015-0899,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2020-06-10,2015-05-13,2016-06-08,2016-07-04,https://github.com/advisories/GHSA-p66x-2cv9-qq3v,published_at,FOREVER_20160608,,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...
2,CVE-2020-15250,junit-team/junit4,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2020-10-12,2020-10-11,2020-10-11,2020-10-12,https://github.com/advisories/GHSA-269g-pwp5-87pp,published_at,r4.13.1,,https://github.com/junit-team/junit4/commit/61...,https://github.com/junit-team/junit4/commit/61...
3,CVE-2020-26259,x-stream/xstream,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2020-12-21,2020-12-12,2020-12-13,2020-12-15,https://github.com/advisories/GHSA-jfvx-7wrx-43fh,published_at,XSTREAM_1_4_15,,https://github.com/x-stream/xstream/commit/0bc...,https://github.com/x-stream/xstream/commit/0bc...
4,CVE-2021-28164,eclipse/jetty.project,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2021-04-06,2021-03-24,2021-03-25,2021-04-01,https://github.com/advisories/GHSA-v7ff-8wcx-gmc5,published_at,jetty-9.4.39.v20210325,,https://github.com/eclipse/jetty.project/commi...,https://github.com/eclipse/jetty.project/commi...
5,CVE-2021-43859,x-stream/xstream,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-02-01,2022-01-29,2022-01-29,2022-02-01,https://github.com/advisories/GHSA-rmr5-cpv2-vgjf,published_at,XSTREAM_1_4_19,,https://github.com/x-stream/xstream/commit/e8e...,https://github.com/x-stream/xstream/commit/e8e...
6,CVE-2022-21724,pgjdbc/pgjdbc,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-02-02,2022-02-01,2022-02-01,2022-02-02,https://github.com/advisories/GHSA-v7wg-cpwc-24m4,published_at,REL42.3.2,,https://github.com/pgjdbc/pgjdbc/commit/f4d0ed...,https://github.com/pgjdbc/pgjdbc/commit/f4d0ed...
7,CVE-2016-4970,netty/netty,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-05-13,2016-06-07,2016-06-07,2017-04-13,https://github.com/advisories/GHSA-rv63-gqm8-9w8q,published_at,netty-4.1.1.Final,,https://github.com/netty/netty/commit/9e2c400f...,https://github.com/netty/netty/commit/9e2c400f...
8,CVE-2016-1181,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-05-13,2016-06-08,2016-06-08,2016-07-04,https://github.com/advisories/GHSA-7jw3-5q4w-89qg,published_at,FOREVER_20160608,,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...
9,CVE-2016-1182,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-05-13,2016-06-08,2016-06-08,2016-07-04,https://github.com/advisories/GHSA-7jw3-5q4w-89qg,published_at,FOREVER_20160608,,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...


In [164]:
# ('Fix', 'Release', 'Disclosure', 'Report'): 1 rows >> remove it 

## Patch Links for Selected Adjusted Lifecycle Order

In [165]:
selected_adjusted_patch_links_df = (
    selected_adjusted_lifecycle_order_df[[
        'CVE_ID',
        'repository',
        'type_of_reporting',
        'First',
        'Second',
        'Third',
        'Fourth',
        'PATCH',
        'PATCH_lifecycle',
    ]]
    .drop_duplicates()
    .reset_index(drop=True)
)

selected_adjusted_patch_links_df

,CVE_ID,repository,type_of_reporting,First,Second,Third,Fourth,PATCH,PATCH_lifecycle
0,CVE-2019-0232,apache/tomcat,github advisory,Fix,Release,Disclosure,GAD,https://github.com/apache/tomcat/commit/4b244d...,https://github.com/apache/tomcat/commit/4b244d...
1,CVE-2015-0899,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,GAD,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...
2,CVE-2020-15250,junit-team/junit4,github advisory,Fix,Release,Disclosure,GAD,https://github.com/junit-team/junit4/commit/61...,https://github.com/junit-team/junit4/commit/61...
3,CVE-2020-26259,x-stream/xstream,github advisory,Fix,Release,Disclosure,GAD,https://github.com/x-stream/xstream/commit/0bc...,https://github.com/x-stream/xstream/commit/0bc...
4,CVE-2021-28164,eclipse/jetty.project,github advisory,Fix,Release,Disclosure,GAD,https://github.com/eclipse/jetty.project/commi...,https://github.com/eclipse/jetty.project/commi...
5,CVE-2021-43859,x-stream/xstream,github advisory,Fix,Release,Disclosure,GAD,https://github.com/x-stream/xstream/commit/e8e...,https://github.com/x-stream/xstream/commit/e8e...
6,CVE-2022-21724,pgjdbc/pgjdbc,github advisory,Fix,Release,Disclosure,GAD,https://github.com/pgjdbc/pgjdbc/commit/f4d0ed...,https://github.com/pgjdbc/pgjdbc/commit/f4d0ed...
7,CVE-2016-4970,netty/netty,github advisory,Fix,Release,Disclosure,GAD,https://github.com/netty/netty/commit/9e2c400f...,https://github.com/netty/netty/commit/9e2c400f...
8,CVE-2016-1181,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,GAD,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...
9,CVE-2016-1182,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,GAD,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...


In [159]:
selected_adjusted_lifecycle_order_df

,CVE_ID,repository,type_of_reporting,Original First,Original Second,Original Third,Original Fourth,First,Second,Third,...,Report Date Day,Fix Date Day,Release Date Day,Disclosure Date Day,reporting_link,created_date_kind,Oldest Tag,manual_day_adjustment_note,PATCH,PATCH_lifecycle
0,CVE-2019-0232,apache/tomcat,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2019-04-18,2019-04-10,2019-04-10,2019-04-15,https://github.com/advisories/GHSA-8vmx-qmch-mpqg,published_at,9.0.18,,https://github.com/apache/tomcat/commit/4b244d...,https://github.com/apache/tomcat/commit/4b244d...
1,CVE-2015-0899,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2020-06-10,2015-05-13,2016-06-08,2016-07-04,https://github.com/advisories/GHSA-p66x-2cv9-qq3v,published_at,FOREVER_20160608,,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...
2,CVE-2020-15250,junit-team/junit4,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2020-10-12,2020-10-11,2020-10-11,2020-10-12,https://github.com/advisories/GHSA-269g-pwp5-87pp,published_at,r4.13.1,,https://github.com/junit-team/junit4/commit/61...,https://github.com/junit-team/junit4/commit/61...
3,CVE-2020-26259,x-stream/xstream,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2020-12-21,2020-12-12,2020-12-13,2020-12-15,https://github.com/advisories/GHSA-jfvx-7wrx-43fh,published_at,XSTREAM_1_4_15,,https://github.com/x-stream/xstream/commit/0bc...,https://github.com/x-stream/xstream/commit/0bc...
4,CVE-2021-28164,eclipse/jetty.project,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2021-04-06,2021-03-24,2021-03-25,2021-04-01,https://github.com/advisories/GHSA-v7ff-8wcx-gmc5,published_at,jetty-9.4.39.v20210325,,https://github.com/eclipse/jetty.project/commi...,https://github.com/eclipse/jetty.project/commi...
5,CVE-2021-43859,x-stream/xstream,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-02-01,2022-01-29,2022-01-29,2022-02-01,https://github.com/advisories/GHSA-rmr5-cpv2-vgjf,published_at,XSTREAM_1_4_19,,https://github.com/x-stream/xstream/commit/e8e...,https://github.com/x-stream/xstream/commit/e8e...
6,CVE-2022-21724,pgjdbc/pgjdbc,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-02-02,2022-02-01,2022-02-01,2022-02-02,https://github.com/advisories/GHSA-v7wg-cpwc-24m4,published_at,REL42.3.2,,https://github.com/pgjdbc/pgjdbc/commit/f4d0ed...,https://github.com/pgjdbc/pgjdbc/commit/f4d0ed...
7,CVE-2016-4970,netty/netty,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-05-13,2016-06-07,2016-06-07,2017-04-13,https://github.com/advisories/GHSA-rv63-gqm8-9w8q,published_at,netty-4.1.1.Final,,https://github.com/netty/netty/commit/9e2c400f...,https://github.com/netty/netty/commit/9e2c400f...
8,CVE-2016-1181,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-05-13,2016-06-08,2016-06-08,2016-07-04,https://github.com/advisories/GHSA-7jw3-5q4w-89qg,published_at,FOREVER_20160608,,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...
9,CVE-2016-1182,kawasima/struts1-forever,github advisory,Fix,Release,Disclosure,Report,Fix,Release,Disclosure,...,2022-05-13,2016-06-08,2016-06-08,2016-07-04,https://github.com/advisories/GHSA-7jw3-5q4w-89qg,published_at,FOREVER_20160608,,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...


In [154]:
# No CSV is saved by default. Uncomment only if you need exports later.
# transparent_lifecycle_order_adjusted_df.to_csv(DATA_DIR / 'transparent_lifecycle_order_adjusted.csv', index=False)
# transparent_lifecycle_summary_adjusted.to_csv(DATA_DIR / 'transparent_lifecycle_order_summary_adjusted.csv', index=False)

In [155]:
selected_adjusted_lifecycle_order_df

,CVE_ID,repository,type_of_reporting,Original First,Original Second,Original Third,Original Fourth,First,Second,Third,...,Report Date Day,Fix Date Day,Release Date Day,Disclosure Date Day,reporting_link,created_date_kind,Oldest Tag,manual_day_adjustment_note,PATCH,PATCH_lifecycle
